# 0. Colab Full Pipeline

This notebook is a literal combined workflow from notebooks 1 through 5. It keeps the same stages in order: setup, debug and sanity, tiny overfit, smoke test, pilot comparison, and final analysis/plots.

Use this if you want one Colab notebook that contains the full research pipeline without switching notebooks. The original modular notebooks are still kept in the repo unchanged.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-mini.git'
REPO_NAME = 'AttnResGPT-mini'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

candidates = [
    Path(f'/content/{REPO_NAME}'),
    Path(f'/content/drive/MyDrive/{REPO_NAME}'),
    Path.cwd(),
]
repo_root = next((p for p in candidates if (p / 'requirements.txt').exists() and (p / 'src').exists()), None)

if repo_root is None:
    target = Path(f'/content/{REPO_NAME}')
    print(f'Cloning {REPO_URL} into {target} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    repo_root = target
else:
    print(f'Using existing repo at {repo_root}')

%cd {repo_root}
!pip -q install -r requirements.txt

In [ ]:
import torch

print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name:', torch.cuda.get_device_name(0))
    print('bf16_supported:', torch.cuda.is_bf16_supported())

# 1. Debug and Sanity

This notebook installs dependencies, finds the repo, checks the GPU, and runs the quick sanity tests plus a small parameter-count and activation-norm inspection.

In [ ]:
!pytest -q tests/test_shapes.py tests/test_masks.py tests/test_forward_pass.py tests/test_attnres.py

In [ ]:
from src.config import load_config
from src.data.dataset import build_dataloaders
from src.eval import build_model
from src.utils import count_parameters

for config_path in ['configs/baseline_t4_small.yaml', 'configs/attnres_t4_small.yaml']:
    cfg = load_config(config_path)
    tokenizer, _, _, _ = build_dataloaders(cfg)
    cfg.model.vocab_size = tokenizer.vocab_size
    model = build_model(cfg)
    print(config_path, count_parameters(model))

In [ ]:
import torch
from src.config import AttnResConfig, ModelConfig
from src.models.gpt_attnres import GPTAttnRes

cfg = ModelConfig(
    architecture='attnres',
    vocab_size=32,
    max_seq_len=32,
    d_model=64,
    n_layers=2,
    n_heads=4,
    d_ff=128,
    dropout=0.0,
    attnres=AttnResConfig(enabled=True, final_readout=True),
)
model = GPTAttnRes(cfg)
input_ids = torch.randint(0, cfg.vocab_size, (2, cfg.max_seq_len))
_, aux = model(input_ids, return_aux=True)
print('block_output_norms:', aux['block_output_norms'])
print('embedding_contribution:', aux['embedding_contribution'])
print('early_contribution:', aux['early_contribution'])
print('late_contribution:', aux['late_contribution'])
print('depth_attention_entropy:', aux['depth_attention_entropy'])

# 2. Tiny Overfit

This notebook runs the tiny overfit pytest check and then launches short baseline and AttnRes training runs on the debug config.

In [ ]:
!pytest -q tests/test_tiny_overfit.py -m slow

In [ ]:
!python -m src.train --config configs/debug_tiny.yaml --overrides experiment.name=nb2_overfit_baseline model.architecture=baseline training.max_steps=150 training.eval_interval=50 training.checkpoint_interval=150
!python -m src.train --config configs/debug_tiny.yaml --overrides experiment.name=nb2_overfit_attnres model.architecture=attnres model.attnres.enabled=true training.max_steps=150 training.eval_interval=50 training.checkpoint_interval=150

In [ ]:
import json
from pathlib import Path

for pattern in ['nb2_overfit_baseline_*', 'nb2_overfit_attnres_*']:
    run_dir = sorted(Path('runs').glob(pattern))[-1]
    summary = json.loads((run_dir / 'run_summary.json').read_text())
    print(run_dir.name)
    print({k: summary[k] for k in ['val_loss', 'val_perplexity', 'best_val_loss'] if k in summary})


# 3. Smoke Test

This notebook runs matched 50-step baseline and AttnRes smoke tests and compares the resulting summaries.

In [ ]:
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb3_smoke_baseline model.architecture=baseline training.max_steps=50 training.eval_interval=25 training.checkpoint_interval=50
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb3_smoke_attnres model.architecture=attnres model.attnres.enabled=true training.max_steps=50 training.eval_interval=25 training.checkpoint_interval=50

In [ ]:
from pathlib import Path

baseline_run = sorted(Path('runs').glob('nb3_smoke_baseline_*'))[-1]
attnres_run = sorted(Path('runs').glob('nb3_smoke_attnres_*'))[-1]
!python scripts/compare_runs.py --baseline-run {baseline_run} --attnres-run {attnres_run}

# 4. Pilot Comparison

This notebook launches matched pilot runs for the baseline and AttnRes models and then compares them with the helper scripts.

In [ ]:
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb4_pilot_baseline model.architecture=baseline
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb4_pilot_attnres model.architecture=attnres model.attnres.enabled=true

In [ ]:
from pathlib import Path

baseline_run = sorted(Path('runs').glob('nb4_pilot_baseline_*'))[-1]
attnres_run = sorted(Path('runs').glob('nb4_pilot_attnres_*'))[-1]
!python scripts/compare_runs.py --baseline-run {baseline_run} --attnres-run {attnres_run}
!python scripts/plot_metrics.py --run-dirs {baseline_run} {attnres_run} --output-dir plots/nb4_pilot

# 5. Analysis and Plots

This notebook finds the latest completed baseline and AttnRes runs, regenerates the comparison plots, and displays the resulting images.

In [ ]:
from pathlib import Path

baseline_candidates = sorted(Path('runs').glob('*baseline*'))
attnres_candidates = sorted(Path('runs').glob('*attnres*'))
if not baseline_candidates or not attnres_candidates:
    raise FileNotFoundError('Run the training notebooks first so there are saved run directories under runs/.')
baseline_run = baseline_candidates[-1]
attnres_run = attnres_candidates[-1]
print('baseline_run:', baseline_run)
print('attnres_run:', attnres_run)
!python scripts/compare_runs.py --baseline-run {baseline_run} --attnres-run {attnres_run}
!python scripts/plot_metrics.py --run-dirs {baseline_run} {attnres_run} --output-dir plots/nb5_analysis

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for image_path in sorted(Path('plots/nb5_analysis').glob('*.png')):
    print(image_path.name)
    display(Image(filename=str(image_path)))